In [1]:
import cv2

def crop_to_ds160(image_path, output_path):
    # Load the image
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Could not load image. Check the file path.")
        return

    # Load OpenCV's built-in Haar Cascade face detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    # Convert to grayscale for detection
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))
    
    if len(faces) == 0:
        print("Error: No face detected in the photo.")
        return
    
    # Assume the primary subject is the largest face detected
    faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)
    x, y, w, h = faces[0]
    
    # DS-160 target: Head height should be roughly 60% (between 50% and 69%) of total height
    target_head_ratio = 0.60
    
    # Calculate the desired dimensions of the final square cropped image
    box_size = int(h / target_head_ratio)
    
    # Center of the detected face box
    center_x = x + w // 2
    center_y = y + h // 2
    
    # Calculate crop coordinates, shifting slightly upward to account for the top of the head
    # (Haar cascade bounding boxes typically go from eyebrows/hairline to chin)
    crop_y1 = int(center_y - (box_size * 0.55)) 
    crop_y2 = crop_y1 + box_size
    crop_x1 = center_x - (box_size // 2)
    crop_x2 = crop_x1 + box_size
    
    # Ensure coordinates stay within the bounds of the original image canvas
    img_h, img_w, _ = img.shape
    
    # Pad image with a white background if the crop area goes outside image boundaries
    pad_top = max(0, -crop_y1)
    pad_bottom = max(0, crop_y2 - img_h)
    pad_left = max(0, -crop_x1)
    pad_right = max(0, crop_x2 - img_w)
    
    if pad_top or pad_bottom or pad_left or pad_right:
        img = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right, 
                                 cv2.BORDER_CONSTANT, value=[255, 255, 255])
        # Readjust coordinates after padding
        crop_y1 += pad_top
        crop_y2 += pad_top
        crop_x1 += pad_left
        crop_x2 += pad_left

    # Crop the final square image
    cropped_img = img[crop_y1:crop_y2, crop_x1:crop_x2]
    
    # Resize to the standard DS-160 digital upload dimension (600x600 pixels)
    final_img = cv2.resize(cropped_img, (600, 600), interpolation=cv2.INTER_CUBIC)
    
    # Save the output file
    cv2.imwrite(output_path, final_img)
    print(f"Success! Cropped DS-160 photo saved to: {output_path}")

In [2]:
# Replace 'photo.jpg' with your actual input file name
input_filename = r"C:\Users\Dani\Downloads\IMG_1721.JPEG"
output_filename = "ds160_ready_photo.jpg"

crop_to_ds160(input_filename, output_filename)

Success! Cropped DS-160 photo saved to: ds160_ready_photo.jpg


In [3]:
import cv2
print(cv2.__version__)
print(hasattr(cv2, 'CascadeClassifier'))

4.13.0
True
